# 04a - raw_count_like の QC（cell / gene 選択のみ）

`guess_preprocessing == "raw_count_like"` の curated h5ad に対して stage1 QC を行う。

## 最重要方針

**QC では細胞と遺伝子を選択するだけ。発現量の値そのものは変更しない。**
stage1 QC 後の h5ad でも `.X` は元の preprocessing state（raw count / raw count-like）の値を保持する。

```
保存する h5ad に対して以下は実行しない（禁止）:
  sc.pp.normalize_total(adata)
  sc.pp.log1p(adata)
  sc.pp.scale(adata)
```

## 入出力

入力:
```
data/curated_h5ad/*.h5ad
results/preprocessing_state/preprocessing_state_summary.csv
```

出力:
```
data/qc_h5ad/raw_count_like/<dataset_id>.stage1_flagged.h5ad   # 全細胞 + QC 列
data/qc_h5ad/raw_count_like/<dataset_id>.stage1_filtered.h5ad  # pass 細胞のみ + gene filter
```

raw count なら raw count のまま保存する。


## セットアップ（パス・定数）

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import matplotlib
matplotlib.use("Agg")  # 非対話バックエンド（保存優先）
import matplotlib.pyplot as plt

# v2 ルートを探す（config/dataset_manifest.yaml がある場所）
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CURATED_DIR = ROOT / "data" / "curated_h5ad"
PREPROC_SUMMARY_CSV = ROOT / "results" / "preprocessing_state" / "preprocessing_state_summary.csv"

PREPROCESSING_STATE = "raw_count_like"
QC_OUT_DIR = ROOT / "data" / "qc_h5ad" / PREPROCESSING_STATE
RESULTS_DIR = ROOT / "results" / "qc_original_scale_pipeline" / "04a_raw_count_like"
PLOT_DIR = RESULTS_DIR / "plots"

for d in (QC_OUT_DIR, RESULTS_DIR, PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

MAX_PLOT = 100_000          # 可視化用サンプリング上限（保存する h5ad は全細胞版）
RNG = np.random.default_rng(0)

print("project root :", ROOT)
print("curated dir  :", CURATED_DIR)
print("qc out dir   :", QC_OUT_DIR)
print("results dir  :", RESULTS_DIR)


project root : /home/suzuki/Learn/SMA/v2
curated dir  : /home/suzuki/Learn/SMA/v2/data/curated_h5ad
qc out dir   : /home/suzuki/Learn/SMA/v2/data/qc_h5ad/raw_count_like
results dir  : /home/suzuki/Learn/SMA/v2/results/qc_original_scale_pipeline/04a_raw_count_like


## stage1 QC 条件（raw_count_like）

```
n_genes_by_counts >= 200
n_genes_by_counts <= 7000
total_counts > 100
pct_counts_mt < 20
gene filter min_cells = 3
```

`qc_reason_stage1`: low_genes / high_genes / low_counts / high_mt


In [2]:
MIN_GENES = 200            # n_genes_by_counts >= 200
MAX_GENES = 7000           # n_genes_by_counts <= 7000
MIN_COUNTS = 100           # total_counts > 100
MAX_PCT_MT = 20            # pct_counts_mt < 20
GENE_FILTER_MIN_CELLS = 3  # sc.pp.filter_genes(adata, min_cells=3)


## gene symbol から mt / ribo フラグを作る

In [3]:
def get_gene_symbols_upper(adata):
    if "gene_symbol_upper" in adata.var.columns:
        return adata.var["gene_symbol_upper"].astype(str).str.upper()
    if "gene_symbol" in adata.var.columns:
        return adata.var["gene_symbol"].astype(str).str.upper()
    return pd.Index(adata.var_names).astype(str).str.upper()


def add_gene_qc_flags(adata):
    gene_symbols = pd.Series(get_gene_symbols_upper(adata), index=adata.var_names)
    adata.var["mt"] = gene_symbols.str.startswith("MT-").values
    adata.var["ribo"] = gene_symbols.str.startswith(("RPS", "RPL", "MRPS", "MRPL")).values
    return adata


## 対象 preprocessing state の dataset を選ぶ

`preprocessing_state_summary.csv` の `guess_preprocessing` でフィルタする。

In [4]:
summary_df = pd.read_csv(PREPROC_SUMMARY_CSV)
target = summary_df[summary_df["guess_preprocessing"].astype(str) == PREPROCESSING_STATE].copy()
print(f"{len(target)} datasets with guess_preprocessing == {PREPROCESSING_STATE!r}\n")
cols = [c for c in ["source_accession", "dataset_id", "guess_preprocessing", "confidence"] if c in target.columns]
print(target[cols].to_string(index=False) if len(target) else "(none)")


9 datasets with guess_preprocessing == 'raw_count_like'

source_accession                                        dataset_id guess_preprocessing confidence
       GSE167198              GSE167198_sc_spinalcord_sod1_dropseq      raw_count_like       high
       GSE167327 GSE167327_sc_spinalcord_optn_cd45_enriched_indrop      raw_count_like       high
       GSE173524               GSE173524_sn_lumbar_spinalcord_sod1      raw_count_like       high
       GSE178693    GSE178693_sc_brainstem_trigeminal_sod1_dropseq      raw_count_like       high
       GSE208629                       GSE208629_sc_spinalcord_sma      raw_count_like       high
       GSE219201          GSE219201_sn_lumbar_spinalcord_sod1_phd1      raw_count_like       high
       GSE242939           GSE242939_sc_spinalcord_sod1_pf04457845      raw_count_like       high
       GSE287569               GSE287569_sc_spinalcord_sod1_ripk1i      raw_count_like       high
       GSE295514                GSE295514_sc_brain_rnls8_tdp4

## QC 関数の定義

`sc.pp.calculate_qc_metrics` で QC 指標を計算し、boolean mask で `qc_pass_stage1` / `qc_reason_stage1` を作る。**`.X` の値は変更しない。**

In [5]:
def compute_qc(adata):
    """calculate_qc_metrics を実行し obs に QC 指標を書き込む。.X は変更しない。"""
    adata = add_gene_qc_flags(adata)
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt", "ribo"],
        percent_top=[50, 100, 200, 500],
        log1p=True,
        inplace=True,
    )
    return adata


In [6]:
def build_stage1_flags(adata):
    """qc_pass_stage1 / qc_reason_stage1 / qc_preprocessing_state を obs に付ける。

    細胞の選択フラグを作るだけで、.X の値は一切変更しない。
    """
    obs = adata.obs

    # boolean mask で細胞条件を明示する
    qc_pass_stage1 = (
        (obs["n_genes_by_counts"] >= MIN_GENES)
        & (obs["n_genes_by_counts"] <= MAX_GENES)
        & (obs["total_counts"] > MIN_COUNTS)
        & (obs["pct_counts_mt"] < MAX_PCT_MT)
    )

    mask_low_genes = obs["n_genes_by_counts"] < MIN_GENES
    mask_high_genes = obs["n_genes_by_counts"] > MAX_GENES
    mask_low_counts = obs["total_counts"] <= MIN_COUNTS
    mask_high_mt = obs["pct_counts_mt"] >= MAX_PCT_MT

    reason = (
        np.where(mask_low_genes, "low_genes,", "")
        + np.where(mask_high_genes, "high_genes,", "")
        + np.where(mask_low_counts, "low_counts,", "")
        + np.where(mask_high_mt, "high_mt,", "")
    )
    reason = pd.Series(reason, index=obs.index).str.rstrip(",")
    reason[reason == ""] = "pass"

    adata.obs["qc_pass_stage1"] = qc_pass_stage1.values
    adata.obs["qc_reason_stage1"] = reason.values
    adata.obs["qc_preprocessing_state"] = PREPROCESSING_STATE
    return adata


## 可視化関数（Scanpy）

In [7]:
def sample_for_plot(adata):
    """巨大データでは可視化用に最大 MAX_PLOT 細胞をサンプリングする。"""
    if adata.n_obs > MAX_PLOT:
        idx = np.sort(RNG.choice(adata.n_obs, size=MAX_PLOT, replace=False))
        return adata[idx].copy()
    return adata


def save_qc_plots(adata, dataset_id):
    """Scanpy の関数で QC 指標を可視化して保存する（全細胞・QC 指標つきの flagged を入力）。"""
    adata_plot = sample_for_plot(adata)
    has_condition = (
        "Condition" in adata_plot.obs.columns
        and adata_plot.obs["Condition"].astype(str).nunique() > 1
    )

    violin_keys = [k for k in [
        "n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo",
        "pct_counts_in_top_50_genes", "pct_counts_in_top_100_genes",
        "pct_counts_in_top_200_genes", "pct_counts_in_top_500_genes",
    ] if k in adata_plot.obs.columns]
    try:
        sc.pl.violin(
            adata_plot,
            keys=violin_keys,
            groupby="Condition" if has_condition else None,
            rotation=90,
            show=False,
        )
        plt.savefig(PLOT_DIR / f"{dataset_id}_violin.png", dpi=100, bbox_inches="tight")
    except Exception as e:
        print(f"  [warn] violin failed: {e}")
    finally:
        plt.close("all")

    for x, y, color in [
        ("total_counts", "n_genes_by_counts", "pct_counts_mt"),
        ("total_counts", "pct_counts_mt", "Condition"),
        ("n_genes_by_counts", "pct_counts_mt", "Condition"),
    ]:
        use_color = None if (color == "Condition" and not has_condition) else color
        try:
            sc.pl.scatter(adata_plot, x=x, y=y, color=use_color, show=False)
            plt.savefig(PLOT_DIR / f"{dataset_id}_scatter_{x}_vs_{y}.png", dpi=100, bbox_inches="tight")
        except Exception as e:
            print(f"  [warn] scatter {x} vs {y} failed: {e}")
        finally:
            plt.close("all")

    try:
        sc.pl.highest_expr_genes(adata_plot, n_top=20, show=False)
        plt.savefig(PLOT_DIR / f"{dataset_id}_highest_expr_genes.png", dpi=100, bbox_inches="tight")
    except Exception as e:
        print(f"  [warn] highest_expr_genes failed: {e}")
    finally:
        plt.close("all")


## dataset ごとに QC を実行

1. curated h5ad を読む
2. `calculate_qc_metrics` で QC 指標を計算
3. `qc_pass_stage1` / `qc_reason_stage1` を作る
4. **stage1_flagged**（全細胞・QC 列追加・`.X` 不変）を保存
5. **stage1_filtered**（pass 細胞のみ・`filter_genes(min_cells=3)`・`.X` 不変）を保存

細胞と遺伝子を選択するだけで、発現量の値そのものは変更しない。

In [8]:
PER_CELL_COLS = [
    "cell_uid", "source_accession", "dataset_id", "Condition", "qc_preprocessing_state",
    "total_counts", "log1p_total_counts", "n_genes_by_counts", "log1p_n_genes_by_counts",
    "pct_counts_mt", "pct_counts_ribo",
    "pct_counts_in_top_50_genes", "pct_counts_in_top_100_genes",
    "pct_counts_in_top_200_genes", "pct_counts_in_top_500_genes",
    "qc_pass_stage1", "qc_reason_stage1",
]

summary_rows = []
per_cell_frames = []

for _, row in target.iterrows():
    dataset_id = str(row["dataset_id"])
    source_accession = str(row.get("source_accession", dataset_id.split("_")[0]))
    h5ad_path = CURATED_DIR / f"{dataset_id}.h5ad"
    if not h5ad_path.exists():
        print(f"[skip] not found: {h5ad_path}")
        continue

    print(f"\n=== {source_accession}  {dataset_id} ===")
    adata = sc.read_h5ad(h5ad_path)
    n_cells_before = adata.n_obs
    n_genes_before = adata.n_vars

    # 1-3. QC 指標計算 + stage1 flag（.X 不変）
    adata = compute_qc(adata)
    adata = build_stage1_flags(adata)
    if "cell_uid" not in adata.obs.columns:
        adata.obs["cell_uid"] = adata.obs_names.astype(str)

    # 4. stage1 flagged 版を保存（全細胞を残し QC 列だけ追加。.X 不変）
    flagged_path = QC_OUT_DIR / f"{dataset_id}.stage1_flagged.h5ad"
    adata.write_h5ad(flagged_path)
    print(f"  flagged  -> {flagged_path.name}  ({adata.n_obs} cells x {adata.n_vars} genes)")

    # 可視化（flagged = 全細胞・QC 指標つき）
    save_qc_plots(adata, dataset_id)

    # 5-6. stage1 filtered 版を保存（qc_pass_stage1==True の細胞だけ。gene filter。.X 不変）
    adata_filt = adata[adata.obs["qc_pass_stage1"].values].copy()
    sc.pp.filter_genes(adata_filt, min_cells=GENE_FILTER_MIN_CELLS)
    n_genes_after = adata_filt.n_vars
    filtered_path = QC_OUT_DIR / f"{dataset_id}.stage1_filtered.h5ad"
    adata_filt.write_h5ad(filtered_path)
    print(f"  filtered -> {filtered_path.name}  ({adata_filt.n_obs} cells x {adata_filt.n_vars} genes)")

    obs = adata.obs
    n_pass = int(obs["qc_pass_stage1"].sum())
    summary_rows.append({
        "source_accession": source_accession,
        "dataset_id": dataset_id,
        "preprocessing_state": PREPROCESSING_STATE,
        "n_cells_before": n_cells_before,
        "n_cells_stage1_pass": n_pass,
        "fraction_stage1_pass": (n_pass / n_cells_before) if n_cells_before else float("nan"),
        "n_genes_before": n_genes_before,
        "n_genes_after_filter_genes_min_cells3": n_genes_after,
        "median_total_counts": float(obs["total_counts"].median()),
        "median_n_genes_by_counts": float(obs["n_genes_by_counts"].median()),
        "median_pct_counts_mt": float(obs["pct_counts_mt"].median()) if "pct_counts_mt" in obs.columns else float("nan"),
        "median_pct_counts_ribo": float(obs["pct_counts_ribo"].median()) if "pct_counts_ribo" in obs.columns else float("nan"),
    })

    pc = pd.DataFrame(index=obs.index)
    for col in PER_CELL_COLS:
        pc[col] = obs[col].values if col in obs.columns else np.nan
    per_cell_frames.append(pc)



=== GSE167198  GSE167198_sc_spinalcord_sod1_dropseq ===


  flagged  -> GSE167198_sc_spinalcord_sod1_dropseq.stage1_flagged.h5ad  (21919 cells x 19777 genes)
  filtered -> GSE167198_sc_spinalcord_sod1_dropseq.stage1_filtered.h5ad  (3355 cells x 14888 genes)

=== GSE167327  GSE167327_sc_spinalcord_optn_cd45_enriched_indrop ===
  flagged  -> GSE167327_sc_spinalcord_optn_cd45_enriched_indrop.stage1_flagged.h5ad  (2125 cells x 33666 genes)
  filtered -> GSE167327_sc_spinalcord_optn_cd45_enriched_indrop.stage1_filtered.h5ad  (1574 cells x 15794 genes)

=== GSE173524  GSE173524_sn_lumbar_spinalcord_sod1 ===
  flagged  -> GSE173524_sn_lumbar_spinalcord_sod1.stage1_flagged.h5ad  (14662 cells x 31040 genes)
  filtered -> GSE173524_sn_lumbar_spinalcord_sod1.stage1_filtered.h5ad  (14076 cells x 24286 genes)

=== GSE178693  GSE178693_sc_brainstem_trigeminal_sod1_dropseq ===
  flagged  -> GSE178693_sc_brainstem_trigeminal_sod1_dropseq.stage1_flagged.h5ad  (15472 cells x 23903 genes)
  filtered -> GSE178693_sc_brainstem_trigeminal_sod1_dropseq.stage1_filte

/tmp/ipykernel_1482423/753165241.py:51: UserWarning: Some cells have zero counts
  sc.pl.highest_expr_genes(adata_plot, n_top=20, show=False)


  filtered -> GSE242939_sc_spinalcord_sod1_pf04457845.stage1_filtered.h5ad  (17351 cells x 21034 genes)

=== GSE287569  GSE287569_sc_spinalcord_sod1_ripk1i ===
  flagged  -> GSE287569_sc_spinalcord_sod1_ripk1i.stage1_flagged.h5ad  (15854 cells x 32285 genes)
  filtered -> GSE287569_sc_spinalcord_sod1_ripk1i.stage1_filtered.h5ad  (13736 cells x 21385 genes)

=== GSE295514  GSE295514_sc_brain_rnls8_tdp43_rds ===
  flagged  -> GSE295514_sc_brain_rnls8_tdp43_rds.stage1_flagged.h5ad  (123401 cells x 24134 genes)
  filtered -> GSE295514_sc_brain_rnls8_tdp43_rds.stage1_filtered.h5ad  (118001 cells x 24055 genes)


## summary CSV と per-cell QC CSV を保存

In [9]:
summary = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / "raw_count_like_qc_summary.csv"
summary.to_csv(summary_path, index=False)
print("saved:", summary_path)

if per_cell_frames:
    per_cell = pd.concat(per_cell_frames, axis=0, ignore_index=True)
else:
    per_cell = pd.DataFrame(columns=PER_CELL_COLS)
per_cell_path = RESULTS_DIR / "raw_count_like_per_cell_qc.csv.gz"
per_cell.to_csv(per_cell_path, index=False, compression="gzip")
print("saved:", per_cell_path)
summary


saved: /home/suzuki/Learn/SMA/v2/results/qc_original_scale_pipeline/04a_raw_count_like/raw_count_like_qc_summary.csv


saved: /home/suzuki/Learn/SMA/v2/results/qc_original_scale_pipeline/04a_raw_count_like/raw_count_like_per_cell_qc.csv.gz


,source_accession,dataset_id,preprocessing_state,n_cells_before,n_cells_stage1_pass,fraction_stage1_pass,n_genes_before,n_genes_after_filter_genes_min_cells3,median_total_counts,median_n_genes_by_counts,median_pct_counts_mt,median_pct_counts_ribo
0,GSE167198,GSE167198_sc_spinalcord_sod1_dropseq,raw_count_like,21919,3355,0.153064,19777,14888,286.0,127.0,25.541126,5.681818
1,GSE167327,GSE167327_sc_spinalcord_optn_cd45_enriched_indrop,raw_count_like,2125,1574,0.740706,33666,15794,3670.0,584.0,0.498812,3.813974
2,GSE173524,GSE173524_sn_lumbar_spinalcord_sod1,raw_count_like,14662,14076,0.960033,31040,24286,9434.0,3177.0,0.000000,0.257852
3,GSE178693,GSE178693_sc_brainstem_trigeminal_sod1_dropseq,raw_count_like,15472,5601,0.362009,23903,19502,243.5,132.0,16.554628,2.380952
4,GSE208629,GSE208629_sc_spinalcord_sma,raw_count_like,52340,39129,0.747593,32285,20910,1510.0,728.5,10.164115,12.728958
5,GSE219201,GSE219201_sn_lumbar_spinalcord_sod1_phd1,raw_count_like,42758,42651,0.997498,32289,25846,2108.5,1310.0,0.176991,1.065375
6,GSE242939,GSE242939_sc_spinalcord_sod1_pf04457845,raw_count_like,1877260,17351,0.009243,32285,21034,1.0,1.0,0.000000,0.000000
7,GSE287569,GSE287569_sc_spinalcord_sod1_ripk1i,raw_count_like,15854,13736,0.866406,32285,21385,7343.0,2716.0,5.287044,7.643786
8,GSE295514,GSE295514_sc_brain_rnls8_tdp43_rds,raw_count_like,123401,118001,0.956240,24134,24055,5661.0,2226.0,4.333782,8.228382


## 次のステップ

`04d_merge_qc_original_scale.ipynb` で、各 preprocessing state の `.stage1_filtered.h5ad` を **正規化せず** に merge する（original-scale merge）。